# 🚗 AutoForensic AI — YOLOv8-Seg Training
**Model:** YOLOv8m-seg (Instance Segmentation)
**Dataset:** CarDD (Car Damage Detection) — 6 classes, oversampled
**GPU:** T4 (15GB VRAM) — Free Colab

### Damage Classes
| ID | Class | Description |
|---|---|---|
| 0 | dent | Body panel dents |
| 1 | scratch | Surface scratches |
| 2 | crack | Structural cracks |
| 3 | glass_shatter | Windshield/window damage |
| 4 | lamp_broken | Headlight/taillight damage |
| 5 | tire_flat | Flat/damaged tires |

## 1. Setup & GPU Check

In [ ]:
# Check GPU availability
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Install Ultralytics
!pip install ultralytics -q

## 2. Mount Google Drive & Setup Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ═══════════════════════════════════════════════════════════
# IMPORTANT: Update this path to where you uploaded
# the CarDD_YOLO_Format folder on your Google Drive
# ═══════════════════════════════════════════════════════════
DRIVE_DATASET = "/content/drive/MyDrive/AutoForensicAI/CarDD_YOLO_Format"

# Verify dataset exists
assert os.path.exists(DRIVE_DATASET), f"Dataset not found at {DRIVE_DATASET}! Update the path above."

# Copy dataset to Colab local storage for faster I/O
LOCAL_DATASET = "/content/CarDD_YOLO_Format"
if not os.path.exists(LOCAL_DATASET):
    print("Copying dataset to local storage (faster I/O)...")
    !cp -r "{DRIVE_DATASET}" /content/
    print("Done!")
else:
    print("Dataset already in local storage.")

# Verify structure
for split in ['train2017', 'val2017', 'test2017']:
    img_count = len(os.listdir(f"{LOCAL_DATASET}/images/{split}"))
    lbl_count = len([f for f in os.listdir(f"{LOCAL_DATASET}/labels/{split}") if f.endswith('.txt')])
    print(f"  {split}: {img_count} images, {lbl_count} labels")

In [ ]:
# Create data.yaml with correct local paths
data_yaml = f"""path: {LOCAL_DATASET}
train: images/train2017
val: images/val2017
test: images/test2017

nc: 6
names: ['dent', 'scratch', 'crack', 'glass_shatter', 'lamp_broken', 'tire_flat']
"""

with open(f"{LOCAL_DATASET}/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created:")
print(data_yaml)

## 3. Train YOLOv8m-Seg

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8m-seg
model = YOLO("yolov8m-seg.pt")

# Train with optimized config for T4
results = model.train(
    # ─── Data ───
    data=f"{LOCAL_DATASET}/data.yaml",
    
    # ─── Training Schedule ───
    epochs=80,
    patience=20,
    batch=12,                   # 12 fits T4. Reduce to 8 if OOM.
    imgsz=640,
    
    # ─── Optimizer ───
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # ─── Augmentation ───
    mosaic=1.0,
    copy_paste=0.3,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    erasing=0.4,
    close_mosaic=10,
    
    # ─── Architecture ───
    overlap_mask=True,
    mask_ratio=4,
    
    # ─── Hardware ───
    device=0,
    workers=2,
    amp=True,
    
    # ─── Saving ───
    save=True,
    save_period=20,
    plots=True,
    val=True,
    
    # ─── Reproducibility ───
    seed=42,
    deterministic=True,
    
    # ─── Project ───
    project="/content/autoforensic_runs",
    name="cardd_yolov8m_seg",
    exist_ok=True,
)

## 4. Evaluate Results

In [ ]:
# Load best model and validate on test set
best_model = YOLO("/content/autoforensic_runs/cardd_yolov8m_seg/weights/best.pt")

# Run validation on test set
metrics = best_model.val(
    data=f"{LOCAL_DATASET}/data.yaml",
    split="test",
    imgsz=640,
    batch=12,
    device=0,
    plots=True,
    save_json=True,
)

print("\n" + "=" * 60)
print("  Test Set Results")
print("=" * 60)
print(f"  Box mAP50:    {metrics.box.map50:.4f}")
print(f"  Box mAP50-95: {metrics.box.map:.4f}")
print(f"  Seg mAP50:    {metrics.seg.map50:.4f}")
print(f"  Seg mAP50-95: {metrics.seg.map:.4f}")
print("=" * 60)

In [ ]:
# Visualize training curves
from IPython.display import Image, display
import os

results_dir = "/content/autoforensic_runs/cardd_yolov8m_seg"

# Show results plot
if os.path.exists(f"{results_dir}/results.png"):
    display(Image(filename=f"{results_dir}/results.png", width=900))

# Show confusion matrix
if os.path.exists(f"{results_dir}/confusion_matrix_normalized.png"):
    display(Image(filename=f"{results_dir}/confusion_matrix_normalized.png", width=700))

# Show prediction samples
for fname in ['val_batch0_pred.jpg', 'val_batch1_pred.jpg']:
    fpath = f"{results_dir}/{fname}"
    if os.path.exists(fpath):
        display(Image(filename=fpath, width=900))

In [ ]:
# Per-class metrics
import pandas as pd

class_names = ['dent', 'scratch', 'crack', 'glass_shatter', 'lamp_broken', 'tire_flat']

print("\nPer-Class Segmentation Metrics:")
print("-" * 60)
print(f"{'Class':>15} {'mAP50':>8} {'mAP50-95':>10} {'Precision':>10} {'Recall':>8}")
print("-" * 60)

for i, name in enumerate(class_names):
    try:
        ap50 = metrics.seg.class_result(i)[2]
        ap = metrics.seg.class_result(i)[3]
        p = metrics.seg.class_result(i)[0]
        r = metrics.seg.class_result(i)[1]
        print(f"{name:>15} {ap50:>8.4f} {ap:>10.4f} {p:>10.4f} {r:>8.4f}")
    except:
        print(f"{name:>15}  -- metrics unavailable --")

print("-" * 60)

## 5. Test Inference on Sample Images

In [ ]:
import glob
from IPython.display import Image, display

# Get 4 random test images
test_images = glob.glob(f"{LOCAL_DATASET}/images/test2017/*.jpg")[:4]

# Run inference
results = best_model.predict(
    source=test_images,
    imgsz=640,
    conf=0.25,
    iou=0.45,
    save=True,
    project="/content/autoforensic_runs",
    name="test_predictions",
    exist_ok=True,
)

# Display predictions
for r in results:
    img_path = r.save_dir / r.path.split('/')[-1] if hasattr(r, 'save_dir') else None
    print(f"\nDetections: {len(r.boxes)} objects found")
    for box, mask in zip(r.boxes, r.masks):
        cls_id = int(box.cls)
        conf = float(box.conf)
        print(f"  - {class_names[cls_id]} ({conf:.2%})")

## 6. Export & Save to Drive

In [ ]:
import shutil

# Create output directory on Drive
DRIVE_OUTPUT = "/content/drive/MyDrive/AutoForensicAI/trained_model"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy best weights to Drive
src_best = "/content/autoforensic_runs/cardd_yolov8m_seg/weights/best.pt"
src_last = "/content/autoforensic_runs/cardd_yolov8m_seg/weights/last.pt"

shutil.copy2(src_best, f"{DRIVE_OUTPUT}/best.pt")
shutil.copy2(src_last, f"{DRIVE_OUTPUT}/last.pt")

# Also copy the full results directory
shutil.copytree(
    "/content/autoforensic_runs/cardd_yolov8m_seg",
    f"{DRIVE_OUTPUT}/full_results",
    dirs_exist_ok=True
)

print(f"\n{'='*60}")
print(f"  Model saved to Google Drive!")
print(f"  Best weights: {DRIVE_OUTPUT}/best.pt")
print(f"  Full results: {DRIVE_OUTPUT}/full_results/")
print(f"{'='*60}")
print(f"\n  Download best.pt to your local project:")
print(f"  D:\\EDAI 07\\models\\best.pt")

In [ ]:
# Export to ONNX for faster CPU inference (optional)
best_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,
    dynamic=False,
)

# Copy ONNX to Drive too
onnx_path = "/content/autoforensic_runs/cardd_yolov8m_seg/weights/best.onnx"
if os.path.exists(onnx_path):
    shutil.copy2(onnx_path, f"{DRIVE_OUTPUT}/best.onnx")
    print(f"ONNX model saved to: {DRIVE_OUTPUT}/best.onnx")